# Taller 01: Búsqueda A* para el 8-puzzle
**Inteligencia Artificial - ELP 8012 - Universidad del Norte**
**Profesor:** Eduardo Zurek, Ph.D.

Este cuaderno implementa el algoritmo de búsqueda A* para resolver el problema del 8-puzzle,
leyendo el estado inicial y el estado final desde los archivos `estadoinicial.txt` y `estadofinal.txt`.

El espacio vacío se representa con el número `0`.


## 1. Representación del estado

El tablero se maneja como una matriz 3x3 de tuplas, por ejemplo:

```
3 2 6
5 0 4
1 8 7
```

Esto en el código se representa como `((3, 2, 6), (5, 0, 4), (1, 8, 7))`. El índice `i` señala la fila y `j` la columna.

## 2. Heurística

La heurística construye pares ordenados entre cada ficha y sus vecinos de la derecha y de abajo.
Si un vecino es `0`, se ignora.

Después compara esas tuplas del estado actual con las del objetivo guardadas en `grafo_objetivo`,
posición por posición.

Ejemplo:
- `grafo_objetivo = [(1, 2), (1, 4), (2, 5)]`
- `grafo_actual = [(1, 2), (1, 4), (2, 3)]`

Aquí solo cambia la última tupla, así que la heurística devuelve `1`.
Mientras más diferencias haya, mayor es el costo estimado.

In [101]:
def construir_grafo(tablero):
    relaciones = []

    for i in range(3):
        for j in range(3):
            valor = tablero[i][j]

            if valor == 0:
                continue

            # Derecha
            if j + 1 < 3:
                vecino = tablero[i][j + 1]
                if vecino != 0:
                    relaciones.append((valor, vecino))

            # Abajo
            if i + 1 < 3:
                vecino = tablero[i + 1][j]
                if vecino != 0:
                    relaciones.append((valor, vecino))

    return relaciones


def heuristica(actual, grafo_objetivo=None):
    """
    Construye tuplas (ficha, vecino) con los vecinos de la derecha y de abajo,
    ignorando el espacio vacío (0), y compara el resultado con el grafo del objetivo.
    """
    grafo_actual = construir_grafo(actual)

    diferencias = 0
    for r1, r2 in zip(grafo_objetivo, grafo_actual):
        if r1 != r2:
            diferencias += 1

    diferencias += abs(len(grafo_objetivo) - len(grafo_actual))
    return diferencias

## 3. Funciones auxiliares

Incluye la lectura de archivos, la impresión de tableros, la verificación de solvabilidad
y la generación de sucesores del estado.

In [102]:
def generar_sucesores(estado):
    """
    Genera los estados sucesores validos moviendo el espacio vacio (0)
    en las 4 direcciones posibles.
    """
    sucesores = []
    fila_vacia = None
    col_vacia = None

    for i in range(3):
        for j in range(3):
            if estado[i][j] == 0:
                fila_vacia = i
                col_vacia = j
                break
        if fila_vacia is not None:
            break

    movimientos = {
        "arriba":    (-1, 0),
        "abajo":     (1, 0),
        "izquierda": (0, -1),
        "derecha":   (0, 1),
    }

    for nombre, (df, dc) in movimientos.items():
        nueva_fila, nueva_col = fila_vacia + df, col_vacia + dc
        if 0 <= nueva_fila < 3 and 0 <= nueva_col < 3:
            nuevo_estado = [list(fila) for fila in estado]
            nuevo_estado[fila_vacia][col_vacia], nuevo_estado[nueva_fila][nueva_col] = \
                nuevo_estado[nueva_fila][nueva_col], nuevo_estado[fila_vacia][col_vacia]
            sucesores.append((tuple(tuple(fila) for fila in nuevo_estado), nombre))

    return sucesores


In [103]:
def leer_estado(nombre_archivo):
    with open(nombre_archivo, "r", encoding="utf-8") as f:
        lineas = [linea.strip() for linea in f if linea.strip()]

    estado = []
    for linea in lineas:
        fila = [int(valor) for valor in linea]
        estado.append(tuple(fila))

    return tuple(estado)


def imprimir_tablero(estado, titulo="Estado:"):
    print(titulo)
    for fila in estado:
        print(" ".join(str(valor) if valor != 0 else "_" for valor in fila))
    print()


def estado_a_tupla(estado):
    return tuple(valor for fila in estado for valor in fila)


def es_resoluble(inicial, objetivo):
    posicion_objetivo = {valor: idx for idx, valor in enumerate(estado_a_tupla(objetivo))}
    permutacion = [posicion_objetivo[v] for v in estado_a_tupla(inicial) if v != 0]

    inversiones = 0
    for i in range(len(permutacion)):
        for j in range(i + 1, len(permutacion)):
            if permutacion[i] > permutacion[j]:
                inversiones += 1

    return inversiones % 2 == 0


## 4. Algoritmo A*

Ejecuta A* desde el estado inicial hasta el objetivo usando la heurística basada en grafos.
Devuelve la ruta, los movimientos y el número de iteraciones.

In [104]:
import heapq
import itertools

def busqueda_a_estrella(inicial, objetivo, heuristica):

    contador = itertools.count()

    g_inicial = 0
    h_inicial = heuristica(inicial, objetivo)
    f_inicial = g_inicial + h_inicial

    frontera = [(f_inicial, next(contador), inicial, g_inicial)]

    padres = {inicial: None}
    movimiento_usado = {inicial: None}
    costo_g = {inicial: 0}

    visitados = set()
    iteraciones = 0
    historial = []

    while frontera:
        f_actual, _, actual, g_actual = heapq.heappop(frontera)

        if actual in visitados:
            continue

        iteraciones += 1
        h_actual = heuristica(actual, objetivo)

        registro = {
            "iteracion": iteraciones,
            "estado_expandido": actual,
            "g": g_actual,
            "h": h_actual,
            "f": f_actual,
            "sucesores_generados": [],
        }

        visitados.add(actual)

        if actual == objetivo:
            historial.append(registro)
            ruta = []
            movimientos = []
            nodo = actual
            while nodo is not None:
                ruta.insert(0, nodo)
                if movimiento_usado[nodo] is not None:
                    movimientos.insert(0, movimiento_usado[nodo])
                nodo = padres[nodo]
            return ruta, movimientos, iteraciones, historial

        for sucesor, nombre_movimiento in generar_sucesores(actual):
            nuevo_g = g_actual + 1
            if sucesor in visitados and nuevo_g >= costo_g.get(sucesor, float("inf")):
                continue
            if nuevo_g < costo_g.get(sucesor, float("inf")):
                costo_g[sucesor] = nuevo_g
                padres[sucesor] = actual
                movimiento_usado[sucesor] = nombre_movimiento
                h_sucesor = heuristica(sucesor, objetivo)
                f_sucesor = nuevo_g + h_sucesor
                heapq.heappush(frontera, (f_sucesor, next(contador), sucesor, nuevo_g))
                registro["sucesores_generados"].append({
                    "movimiento": nombre_movimiento,
                    "estado": sucesor,
                    "g": nuevo_g,
                    "h": h_sucesor,
                    "f": f_sucesor,
                })

        historial.append(registro)

    return None, None, iteraciones, historial


## 5. Programa principal

Lee los archivos de entrada, verifica si el problema tiene solución y muestra el número
de iteraciones y el recorrido completo hasta el estado final.

In [105]:
# Lectura de los archivos de entrada
inicial = leer_estado("estadoinicial.txt")
objetivo = leer_estado("estadofinal.txt")

grafo_objetivo = construir_grafo(objetivo)

imprimir_tablero(inicial, "Estado Inicial:")
imprimir_tablero(objetivo, "Estado Objetivo:")

if not es_resoluble(inicial, objetivo):
    print("Este problema NO tiene solucion: el estado objetivo no es alcanzable "
          "desde el estado inicial dado.")
else:
    ruta, movimientos, iteraciones, historial = busqueda_a_estrella(
        inicial, objetivo, heuristica
    )
    print(f"Numero de iteraciones realizadas: {iteraciones}")
    print(f"Numero de movimientos de la solucion: {len(movimientos)}")
    print()
    for i, estado in enumerate(ruta):
        titulo = f"Paso {i}" if i > 0 else "Estado Inicial"
        if i > 0:
            titulo += f"  (movimiento: {movimientos[i-1]})"
        print(titulo)
        for fila in estado:
            print(" ".join(str(v) if v != 0 else "_" for v in fila))
        print()

Estado Inicial:
3 2 6
5 _ 4
1 8 7

Estado Objetivo:
8 4 7
2 6 5
3 1 _

Numero de iteraciones realizadas: 61782
Numero de movimientos de la solucion: 20

Estado Inicial
3 2 6
5 _ 4
1 8 7

Paso 1  (movimiento: abajo)
3 2 6
5 8 4
1 _ 7

Paso 2  (movimiento: izquierda)
3 2 6
5 8 4
_ 1 7

Paso 3  (movimiento: arriba)
3 2 6
_ 8 4
5 1 7

Paso 4  (movimiento: arriba)
_ 2 6
3 8 4
5 1 7

Paso 5  (movimiento: derecha)
2 _ 6
3 8 4
5 1 7

Paso 6  (movimiento: abajo)
2 8 6
3 _ 4
5 1 7

Paso 7  (movimiento: abajo)
2 8 6
3 1 4
5 _ 7

Paso 8  (movimiento: izquierda)
2 8 6
3 1 4
_ 5 7

Paso 9  (movimiento: arriba)
2 8 6
_ 1 4
3 5 7

Paso 10  (movimiento: arriba)
_ 8 6
2 1 4
3 5 7

Paso 11  (movimiento: derecha)
8 _ 6
2 1 4
3 5 7

Paso 12  (movimiento: derecha)
8 6 _
2 1 4
3 5 7

Paso 13  (movimiento: abajo)
8 6 4
2 1 _
3 5 7

Paso 14  (movimiento: abajo)
8 6 4
2 1 7
3 5 _

Paso 15  (movimiento: izquierda)
8 6 4
2 1 7
3 _ 5

Paso 16  (movimiento: arriba)
8 6 4
2 _ 7
3 1 5

Paso 17  (movimiento: arriba)
8